[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/orientation/conformal_circle/conformal_circle.ipynb)

# A circle constructed twice

A circle inherits an orientation from the order of three points on it, or from the order and chosen sides of two intersecting surfaces. Reflecting the circle can distinguish these constructions even when their coefficients agree.

Conformal geometric algebra holds spheres and planes as vectors, points as antivectors, and both constructions of a circle as bivectors.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np
from numga import Algebra, NumpyContext, stack
from examples.animation import save_figure, show_animation
from examples.orientation.conformal_circle import render
from examples.orientation.conformal_circle.core import circle_samples, disk_samples, sphere_samples

In [ ]:
ga = Algebra("x+y+z+w+e-")
mv = NumpyContext(ga).multivector
# + extrinsic, - intrinsic
Point = ga.gatype.antivector()                # Antivector-
Sphere = ga.gatype.vector()                   # Vector+
Circle = ga.gatype.bivector()                 # Bivector+
Path = Circle.derive.flipped_orientation      # Bivector-

The unit sphere meets the horizontal plane in the same circle that passes through the three ordered points. Reflection in that plane fixes every circle point and preserves their order. It also fixes the sphere, but reverses the plane's chosen side.

In [ ]:
points: Point = stack([(mv.x + mv.e).dual(), (-mv.y + mv.e).dual(), (-mv.x + mv.e).dual()])  # Antivector-
plane: Sphere = mv.z                                      # Vector+
sphere: Sphere = mv.w                                     # Vector+
mirror: Sphere = plane                                    # Vector+
samples = 96
meridians = 16
parallels = 9
angles = np.linspace(0, 2 * np.pi, samples + 1)
longitudes = np.linspace(0, 2 * np.pi, meridians + 1)
latitudes = np.linspace(-np.pi / 2, np.pi / 2, parallels)

# Joining the ordered points and intersecting the surfaces produce the same blade coefficients.
through_points: Path = (points[0] & points[1] & points[2]).normalized()  # Bivector-
from_surfaces: Circle = (plane ^ sphere).normalized()     # Bivector+

# Reflect each construction; their orientation types determine the action.
reflected_path: Path = mirror >> through_points           # Bivector-
reflected_cut: Circle = mirror >> from_surfaces           # Bivector+

# The display samples the sphere and circle, then mirrors the entire construction.
curve: Point = circle_samples(points[0], mv.xy, angles)   # Antivector-
surface: Point = sphere_samples(points[0], mv.xy, mv.xz, longitudes, latitudes)  # Antivector-
vertices: Point = stack([points, mirror >> points])       # Antivector-
curves: Point = stack([curve, mirror >> curve])           # Antivector-
surfaces: Point = stack([surface, mirror >> surface])     # Antivector-
planes: Sphere = stack([plane, mirror >> plane])          # Vector+
spheres: Sphere = stack([sphere, mirror >> sphere])       # Vector+

np.set_printoptions(precision=3, suppress=True)
print("Through points:\n" + through_points.coefficient_table())
print("From surfaces:\n" + from_surfaces.coefficient_table())
print("Through points, reflected:\n" + reflected_path.coefficient_table())
print("From surfaces, reflected:\n" + reflected_cut.coefficient_table())
save_figure(render.draw_circle(vertices, curves, surfaces, planes, spheres), "orientation_conformal_circle");

The left panel is the construction before reflection; the right is its mirror. The dark circle arrows and three coloured markers retain their order. The orange arrow still points into the sphere, while the teal arrow changes the plane's positive side. Thus the point join remains `+zw`, but the surface intersection becomes `−zw`. Rebuilding either circle from its reflected ingredients gives the corresponding reflected result.

Points can travel along this circle in different ways. On the left, they rotate rigidly about its axis. On the right, a reflection combined with a conformal boost moves them unevenly along the circle, fixing its top and bottom points. This is a [hyperbolic glide reflection](https://hamishtodd1.substack.com/p/hyperbolic-glide-reflections). The intrinsic circle is not itself exponentiated: its dual, paired with infinity or a reflecting plane, supplies an extrinsic bivector generator.

In [ ]:
infinity: Sphere = mv.e + mv.w                            # Vector+
origin: Sphere = (mv.e - mv.w) / 2                        # Vector+
reflector: Sphere = mv.x                                  # Vector+
frames = 64
radial_samples = 40
angular_samples = 96
glide_amplitude = 0.9
duration_ms = 70
phase = np.linspace(0, 2 * np.pi, frames, endpoint=False)
radii = np.linspace(0, 1, radial_samples + 1)
angles = np.linspace(0, 2 * np.pi, angular_samples + 1)

# Infinity selects the Euclidean rotation about the circle's axis.
axis: Circle = (through_points.dual() | infinity).normalized()  # Bivector+
rotations = (axis * (-phase / 2)).exp()                   # Even+

# The reflection commutes with this boost; zero boost still leaves the reflection.
boost: Circle = reflector | through_points.dual()         # Bivector+
glides = reflector * (boost * (glide_amplitude * np.sin(phase))).exp()  # Odd+

# A grid inside the circle makes rigidity and conformal distortion visible.
grid: Point = disk_samples(mv.x, origin, infinity, axis, radii, angles)  # Antivector-
moving_grid: Point = stack([rotations[:, None, None] >> grid,
                           glides[:, None, None] >> grid], axis=1)  # Antivector-
fixed_points: Point = stack([(mv.y + mv.e).dual(), (-mv.y + mv.e).dual()])  # Antivector-

show_animation(render.animate_circle_motions(moving_grid, fixed_points),
               "orientation_circle_motions", duration_ms)

The coloured markers follow the same sampled points in both panels. The left grid keeps its shape; the right grid bends and crowds toward one of the two ringed fixed points. The boost sweeps back and forth while the reflection stays present, reversing the marker order. Exponentiating the surface intersection `from_surfaces` instead would fix every point on the circle and move surrounding points around it.